In [1]:
import time as tm
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import schwabdev as sd
import time as tm
import zarr
import shutil
import os

from datetime import date, time, datetime, timedelta

import warnings

warnings.filterwarnings('ignore', message="The data type .* does not have a Zarr V3 specification.*")
warnings.filterwarnings('ignore', message="Consolidated metadata is currently not part in the Zarr format 3 specification.*")

from main_classes.DataManager import DataManager as DM
from utility.lib_apiManagment import create_client as CC


dm = DM()
zarr_store = xr.open_dataset(dm.hot_db_path)
zarr_store

<xarray.Dataset> Size: 21GB
Dimensions:  (day: 49, time: 288, ident: 5094, qVar: 36, fVar: 18)
Coordinates:
  * day      (day) <U10 2kB '2025-09-14' '2025-09-15' ... '2025-11-11'
  * ident    (ident) <U5 102kB 'A' 'AA' 'AACB' 'AACI' ... 'ZYBT' 'ZYME' 'ZYXI'
  * time     (time) <U5 6kB '00:00' '00:05' '00:10' ... '23:45' '23:50' '23:55'
  * fVar     (fVar) <U28 2kB 'assetSubType' 'ssid' ... 'quote.closePrice'
  * qVar     (qVar) <U29 4kB 'reference.htbRate' ... 'quote.postMarketPercent...
Data variables:
    5m       (day, time, ident, qVar) float64 21GB ...
    1d       (day, ident, fVar) float64 36MB ...

In [27]:
DATES = ['2025-09-22','2025-09-23','2025-09-24','2025-09-25','2025-09-26','2025-09-29','2025-09-30','2025-10-01','2025-10-02','2025-10-03']
TIMES = list(pd.date_range("09:30","15:30",freq='5min').strftime("%H:%M"))
TIMES2 = list(pd.date_range("10:00","15:30",freq='5min').strftime("%H:%M"))

final_pairs = []

for date in DATES:
    d_vol10 = zarr_store['1d'].sel(day=date,fVar='fundamental.avg10DaysVolume')
    d_exch = zarr_store['1d'].sel(day=date,fVar='reference.exchange')

    df_vol10 = d_vol10.to_pandas()
    df_exch = d_exch.to_pandas()

    d_open = zarr_store['5m'].sel(day=date,qVar='quote.openPrice',time='10:00')
    df_open = d_open.to_pandas()

    df_5m_raw = zarr_store['5m'].sel(day=date,qVar='quote.mark',time=TIMES).to_pandas()

    df_5m_o1 = df_5m_raw.pct_change(fill_method=None).loc[TIMES2]
    df_5m_o2 = df_5m_raw.pct_change(fill_method=None,periods=2).loc[TIMES2]
    df_5m_o6 = df_5m_raw.pct_change(fill_method=None,periods=6).loc[TIMES2]

    df_5m_o1['mr'] = df_5m_o1.mean(axis=1)
    df_5m_o2['mr'] = df_5m_o2.mean(axis=1)
    df_5m_o6['mr'] = df_5m_o6.mean(axis=1)

    o1_mr = df_5m_o1['mr'].values
    o2_mr = df_5m_o2['mr'].values
    o6_mr = df_5m_o6['mr'].values

    df_5m_o1_xm = df_5m_o1.sub(df_5m_o1['mr'],axis=0).drop('mr',axis=1)
    df_5m_o2_xm = df_5m_o2.sub(df_5m_o2['mr'],axis=0).drop('mr',axis=1)
    df_5m_o6_xm = df_5m_o6.sub(df_5m_o6['mr'],axis=0).drop('mr',axis=1)
    
    for stock in df_5m_o1_xm.columns:
        stock_5m_o1_xm = df_5m_o1_xm[stock].values
        stock_5m_o2_xm = df_5m_o2_xm[stock].values
        stock_5m_o6_xm = df_5m_o6_xm[stock].values
        for i in range(len(stock_5m_o1_xm) - 1):
            final_pairs.append({
                'stock': stock,
                'date': date,
                'time': TIMES2[i],
                'rxm_t..t+1': stock_5m_o1_xm[i+1],
                'rxm_t-1..t': stock_5m_o1_xm[i],
                'rxm_t-2..t': stock_5m_o2_xm[i],
                'rxm_t-6..t': stock_5m_o6_xm[i],
                'mr_t-1..t': o1_mr[i],
                'mr_t-2..t': o2_mr[i],
                'mr_t-6..t': o6_mr[i],
                'avg_vol_10': df_vol10[stock],
                'exchange': df_exch[stock],
                'above_open': (1 if df_5m_raw[stock].values[i] > df_open[stock] else 0),
                'last_price': df_5m_raw[stock].values[i],
                
            })

result_df = pd.DataFrame(final_pairs)

In [28]:
result_df

,stock,date,time,rxm_t..t+1,rxm_t-1..t,rxm_t-2..t,rxm_t-6..t,mr_t-1..t,mr_t-2..t,mr_t-6..t,avg_vol_10,exchange,above_open
0,A,2025-09-22,10:00,0.000562,0.001967,0.001072,0.002544,0.001576,0.003633,0.001520,1808670.0,0.0,1
1,A,2025-09-22,10:05,-0.003646,0.000562,0.002540,-0.002962,-0.000443,0.001124,0.003557,1808670.0,0.0,1
2,A,2025-09-22,10:10,0.000373,-0.003646,-0.003086,-0.004214,0.000711,0.000269,0.005768,1808670.0,0.0,0
3,A,2025-09-22,10:15,-0.001906,0.000373,-0.003254,-0.002861,-0.000334,0.000358,0.005511,1808670.0,0.0,0
4,A,2025-09-22,10:20,-0.000279,-0.001906,-0.001521,-0.003524,-0.000481,-0.000827,0.003047,1808670.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3362035,ZYXI,2025-10-03,15:05,-0.000375,-0.000049,-0.000509,-0.006865,0.000049,0.000509,0.000016,81771.0,4.0,1
3362036,ZYXI,2025-10-03,15:10,-0.000422,-0.000375,-0.000423,-0.007204,0.000375,0.000423,0.000354,81771.0,4.0,1
3362037,ZYXI,2025-10-03,15:15,0.013975,-0.000422,-0.000794,-0.000844,0.000422,0.000794,0.000844,81771.0,4.0,1
3362038,ZYXI,2025-10-03,15:20,-0.000012,0.013975,0.013557,-0.000901,-0.000182,0.000236,0.001173,81771.0,4.0,1


In [29]:
result_df.to_csv('ECON_211_Project_Data_Set_v1.csv')